# SpectraShift Week 9: aggregate representation and robustness analysis
Use CPU with Internet off. Attach source v8, Week 8 complete, Week 9 contracts, Week 9 probes, and all three Week 9 diagnostic datasets.


In [ ]:
from pathlib import Path
import hashlib, json, os, shutil, sys, yaml

INPUT = Path('/kaggle/input')
projects = [p.parent for p in INPUT.rglob('pyproject.toml') if (p.parent / 'src/spectrashift/train/week9.py').is_file()]
if not projects:
    bundles = sorted(INPUT.rglob('spectrashift-kaggle-source.zip'))
    assert len(bundles) == 1, f'Expected one Week 9 source bundle, found {bundles}'
    source_work = Path('/tmp/spectrashift-week9-source')
    if source_work.exists(): shutil.rmtree(source_work)
    shutil.unpack_archive(str(bundles[0]), str(source_work))
    projects = [source_work]
assert projects, 'No Week 9 source tree found'
PROJECT = sorted(projects, key=lambda path: len(str(path)))[0]
sys.path.insert(0, str(PROJECT / 'src'))
os.chdir(PROJECT)

def unique_file(name):
    candidates = sorted(INPUT.rglob(name))
    by_hash = {}
    for path in candidates:
        by_hash.setdefault(hashlib.sha256(path.read_bytes()).hexdigest(), path)
    assert len(by_hash) == 1, f'Expected one unique {name}; found {candidates}'
    return next(iter(by_hash.values()))

WORK = Path('/kaggle/working/spectrashift-week9-complete')
WORK.mkdir(parents=True, exist_ok=True)
WEEK9_CONTRACTS = unique_file('week9_contracts_summary.json')
WEEK9_CONTRACT = unique_file('week9_contract.json')
EVAL_LABELS = unique_file('evaluation_labels.parquet')
SUPPORT_CONTRACT = unique_file('support_contract.json')
PROBES = unique_file('week9_probe_summary.json')
diagnostic_summaries = []
for seed in (17, 29, 43):
    matches = sorted(INPUT.rglob(f'week9_diagnostics_seed{seed}_summary.json'))
    assert len(matches) == 1, f'Expected one seed {seed} diagnostic summary, found {matches}'
    diagnostic_summaries.append(matches[0])
config = yaml.safe_load((PROJECT / 'configs/analysis/week9.yaml').read_text())
config['paths'].update({
    'week9_contracts_summary_path': str(WEEK9_CONTRACTS),
    'week9_contract_path': str(WEEK9_CONTRACT), 'evaluation_labels_path': str(EVAL_LABELS),
    'support_contract_path': str(SUPPORT_CONTRACT),
})
RUNTIME_CONFIG = WORK / 'week9.yaml'
RUNTIME_CONFIG.write_text(yaml.safe_dump(config, sort_keys=False))


In [ ]:
from spectrashift.train.week9 import aggregate_week9
summary = aggregate_week9(RUNTIME_CONFIG, PROBES, diagnostic_summaries, WORK)
print(json.dumps(summary, indent=2))
assert summary['week9_complete'] and summary['week10_approved']
assert summary['linear_probe_count'] == 108 and summary['knn_probe_count'] == 18
assert summary['stress_prediction_domain_count'] == 108
assert summary['nearest_neighbor_row_count'] == 3000
assert summary['model_selection_after_week8'] is False
assert summary['encoder_updates_during_week9'] is False
assert summary['week10_implemented'] is False
